### Start by loading in data

- UN text data
- GDP data
- climate associated words data

In [2]:
import os

"""
GETTING THE TEXT DATA FROM FOLDER DATA/TXT
"""



def get_text_data(output=False, N=False):
    """
    gets all the text data from the subfolders, can print it and you can limit the number of prints
    returns a dictionary with key the title of the file and value the text of the file
    """
    subfolders = []
    for folder in os.listdir('data/TXT'):
        if os.path.isdir(os.path.join('data/TXT', folder)):
            subfolders.append(folder)

    text_data = {}
    for folder in subfolders:
        for file in os.listdir(os.path.join('data/TXT', folder)):
            if file.endswith('.txt'):
                with open(os.path.join('data/TXT', folder, file), 'r') as f:
                    text_data[file] = f.read()
                    if output:
                        print(f"Title: {file}")
                        print(f"Text: {text_data[file]}")
                        print("\n")
                    if N and len(text_data) >= N:
                        return text_data
    return text_data

#test with first file
text_data = get_text_data()
print(text_data["ARG_01_1946.txt"])


At the resumption of the first session of the General Assembly, the Argentine delegation wishes to state its views on a number of questions.
Politics are determined by circumstances. Accordingly, in making these remarks, we do not renounce the right to take decisions in the light of events; but we nevertheless think it desirable to state here certain useful principles to serve as a guide for our actions. We hope that these remarks will be interpreted at their true value. We are not sceptics, but our relative optimism would be strengthened if we found that a spirit of conciliation were gaining ground. Otherwise, we should be obliged to resume complete freedom of action, and proceed in accordance with the circumstances and the interests of the country which we represent.
We are no friends of unanimity, which is wont to disguise undesirable, if not improper pressure. Even the hottest debates in the defence of conflicting points of view are not incompatible with the desire to reach a solut

In [17]:
import keyword
import regex as re

climate_words_dict = {"greenhouse_gases_and_emissions": [
        "carbon dioxide", "co2", "methane", "ch4", "nitrous oxide", "n2o",
        "fluorinated gases", "greenhouse effect", "carbon emissions",
        "scope 1 emissions", "scope 2 emissions", "scope 3 emissions",
        "carbon footprint", "carbon budget", "emission intensity", "flaring"
    ],
    "climate_science_and_phenomena": [
        "global warming", "surface temperature", "temperature anomaly",
        "radiative forcing", "albedo effect", "feedback loop", "tipping point",
        "keeling curve", "ocean acidification", "paleoclimatology", "antarctic ice sheet",
        "arctic sea ice", "permafrost thawing", "glacier retreat", "thermohaline circulation",
        "el nino", "la nina", "enso", "polar vortex"
    ],
    "extreme_weather_and_impacts": [
        "sea level rise", "storm surge", "coastal erosion", "flash drought",
        "megadrought", "heatwave", "heat dome", "wildfire", "forest fire",
        "tropical cyclone", "typhoon", "hurricane intensification",
        "biodiversity loss", "coral bleaching", "species extinction",
        "habitat fragmentation", "vector-borne disease", "water scarcity"
    ],
    "energy_and_technology": [
        "clean energy", "renewable energy", "solar photovoltaic", "wind turbine",
        "geothermal", "hydroelectric", "nuclear energy", "grid storage",
        "battery storage", "green hydrogen", "electrification", "heat pump",
        "electric vehicles", "ev adoption", "carbon capture and storage", "ccs",
        "direct air capture", "dac", "bioenergy"
    ],
    "policy_agreements_and_economics": [
        "ipcc", "cop28", "cop29", "cop30", "paris agreement", "unfccc",
        "nationally determined contributions", "ndc", "carbon pricing",
        "carbon tax", "cap and trade", "emissions trading system", "ets",
        "esg", "green finance", "loss and damage", "carbon border adjustment mechanism",
        "cbam", "net zero", "carbon neutrality", "climate justice"
    ],
    "adaptation_and_nature_solutions": [
        "climate adaptation", "climate resilience", "afforestation",
        "reforestation", "blue carbon", "mangrove restoration",
        "regenerative agriculture", "soil carbon sequestration",
        "rewilding", "sponge cities", "flood defense", "sea wall"
    ], 
    "school": [
        "school", "education", "curriculum", "teacher", "student",
        "classroom", "lesson", "learning", "teaching", "pedagogy", "university", "college", "schooling", "academic", "syllabus",
        "schooling", "academic", "syllabus", "instruction", "training",
        "schoolwork", "homework", "assignment", "exam", "test", "assessment", "grading", "evaluation", "classroom management",
        "school policy", "school"]
}

 


In [16]:
# load GDP.csv data 
import pandas as pd
gdp = pd.read_csv("data/GDP.csv")
gdp

# list_of_countries = gdp['Country'].unique().tolist()


,Country,Country Code,1960,1961,1962,1963,1964,1965,1966,1967,...,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
0,Aruba,ABW,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.727933e+09,2.791061e+09,2.963128e+09,2.983799e+09,3.092179e+09,3.276188e+09,3.395794e+09,2.610039e+09,3.126019e+09,NaN
1,Africa Eastern and Southern,AFE,2.112502e+10,2.161623e+10,2.350628e+10,2.804836e+10,2.592067e+10,2.947210e+10,3.201437e+10,3.326951e+10,...,9.859871e+11,1.006526e+12,9.273485e+11,8.851764e+11,1.021043e+12,1.007196e+12,1.000834e+12,9.275933e+11,1.081998e+12,1.169484e+12
2,Afghanistan,AFG,5.377778e+08,5.488889e+08,5.466667e+08,7.511112e+08,8.000000e+08,1.006667e+09,1.400000e+09,1.673333e+09,...,2.056449e+10,2.055058e+10,1.999814e+10,1.801955e+10,1.889635e+10,1.841886e+10,1.890450e+10,2.014345e+10,1.458314e+10,NaN
3,Africa Western and Central,AFW,1.044764e+10,1.117321e+10,1.199053e+10,1.272769e+10,1.389811e+10,1.492979e+10,1.591084e+10,1.451058e+10,...,8.339481e+11,8.943225e+11,7.686447e+11,6.913634e+11,6.848988e+11,7.670257e+11,8.225384e+11,7.864600e+11,8.444597e+11,8.778633e+11
4,Angola,AGO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.334016e+11,1.372444e+11,8.721930e+10,4.984049e+10,6.897277e+10,7.779294e+10,6.930911e+10,5.024137e+10,6.568544e+10,1.067136e+11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,Kosovo,XKX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,6.735328e+09,7.074393e+09,6.295845e+09,6.682674e+09,7.180769e+09,7.878763e+09,7.899741e+09,7.717143e+09,9.412034e+09,9.429156e+09
262,"Yemen, Rep.",YEM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.041523e+10,4.322859e+10,4.244449e+10,3.131782e+10,2.684223e+10,2.160616e+10,NaN,NaN,NaN,NaN
263,South Africa,ZAF,8.748597e+09,9.225996e+09,9.813996e+09,1.085420e+10,1.195600e+10,1.306899e+10,1.421139e+10,1.582139e+10,...,4.008860e+11,3.811989e+11,3.467098e+11,3.235855e+11,3.814488e+11,4.041589e+11,3.885312e+11,3.376196e+11,4.190156e+11,4.058697e+11
264,Zambia,ZMB,7.130000e+08,6.962857e+08,6.931429e+08,7.187143e+08,8.394286e+08,1.082857e+09,1.264286e+09,1.368000e+09,...,2.803724e+10,2.714102e+10,2.125122e+10,2.095841e+10,2.587360e+10,2.631151e+10,2.330867e+10,1.811064e+10,2.214765e+10,2.978445e+10


In [62]:
#load pisa scores
pisa_df = pd.read_csv("data/average-performance-of-15-year-old-students-by-subject.csv")
pisa_df["average"] = pisa_df[["Mathematics", "Reading", "Science"]].mean(axis=1)
pisa_df_country = pisa_df.groupby("Entity").sum()
# display(pisa_df[pisa_df["Entity"]=="United States"], pisa_df[pisa_df["Entity"]=="Albania"])

pisa_df_filtered = pisa_df.groupby("Entity").filter(lambda g: len(g) > 7)
pisa_df_per_country = pisa_df_filtered.groupby(["Entity", "Year"]).sum()
# pisa_df_per_country
pisa_df_per_country

Code  Mathematics    Science    Reading     average
Entity        Year                                                    
Australia     2000  AUS      0.00000    0.00000  528.27850  528.278500
              2003  AUS    524.26600    0.00000  525.42700  524.846500
              2006  AUS    519.90780  526.87960  512.89330  519.893567
              2009  AUS    514.34045  527.27050  514.90063  518.837193
              2012  AUS    504.15076  521.49475  511.80400  512.483170
...                 ...          ...        ...        ...         ...
United States 2009  USA    487.39650  502.00226  499.82680  496.408520
              2012  USA    481.36680  497.40982  497.58173  492.119450
              2015  USA    469.62848  496.24243  496.93510  487.602003
              2018  USA    478.24472  502.38004  505.35278  495.325847
              2022  USA    464.88803  499.41406  503.93756  489.413217

[264 rows x 5 columns]

## EDA
from the dataframe of PISA scores, we can for example take Albania and USA. Albania does not have a record for each 3 years

## clean the text data and first scan

### speech data
---
We will use nltk module "stopwords" to remove stop words from the text data. 
The analysis will mainly focus on finding certain words in the text. By removing stop words, we do not have to match those words against words we are looking for. This will speed up the analysis. 
Furthermore, when doing a broader sentiment analysis, stop words will not be usefull and only make the text less information dense.

After removing stop words, we will quicky count the number of words per speech per country. This gives an indication of how long the speeches are and if they differ in lenght. This information is not needed for the final analysis, but helps with understanding the data.

### PISA score data
---
there are 88 countries in the PISA score data. If we only look at countries with entries from 2003 to 2022, there are 31 remaining. It is not needed to drop those countries, since we will predict PISA based on speeches from that country the year before. 

In [ ]:
import nltk
nltk.download('stopwords')

def speech_set(text_data):
    """
    replaces dict value with set instead of list
    """
    for key in text_data:
        text_data[key] = set(text_data[key].split())
    return text_data

def speech_set_no_stopwords(text_data):
    """
    removes stopwords (input should be set so call speech_set first)
    """
    stopwords = set(nltk.corpus.stopwords.words('english'))
    
    for key in text_data:
        text_data[key] = text_data[key] - stopwords
    return text_data



text = speech_set(text_data)
text = speech_set_no_stopwords(text)



[nltk_data] Downloading package stopwords to /Users/jacob/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:
# each key starts with 3 letter code. 
# generate new dict with key as 3 letter code and as value the number of words that appear in text (keys can have same 3 letter as beginning, so add those to the total)
word_count = {}
for key in text:
    token = key[:3]
    if token not in word_count:
        word_count[token] = []
    word_count[token].append(len(text[key]))

for key in word_count:
    word_count[key] = sum(word_count[key])/len(word_count[key])

sorted_word_count = sorted(word_count.items(), key=lambda x:x[1], reverse=True)

### counting the number of mentions of climate associated words
Combining the data from the UN speeches and the climate associated words, we can count the number of mentions of climate associated words per speech per country per year. We will store it by country, topic and year. 

In [18]:
from turtle import rt
import pandas as pd
import re

records = []

for country_speech_year, speech_tokens in text.items():

    if "DS_Store" in country_speech_year:
        continue  
    
    country = country_speech_year[:3]
    year = int(country_speech_year[-8:-4])
   
    speech_str = " ".join(speech_tokens)
    
    for topic, word_list in climate_words_dict.items():
        pattern = re.compile(r"\b(" + "|".join(re.escape(w) for w in word_list) + r")\b", re.IGNORECASE)
        matches = pattern.findall(speech_str)
        
        if matches:
            records.append({
                "country": country,
                "year": year,
                "topic": topic,
                "mention_count": len(matches),
                "matched_words": list(set(m.lower() for m in matches))
            })

df_climate = pd.DataFrame(records)

In [20]:
df_climate[df_climate["topic"]=="school"]

,country,year,topic,mention_count,matched_words
0,BEL,1950,school,1,[lesson]
1,FRA,1950,school,1,[test]
2,TUR,1950,school,2,[test]
3,BRA,1950,school,1,[education]
4,DOM,1950,school,2,[lesson]
...,...,...,...,...,...
7526,VEN,2014,school,1,[education]
7527,LIE,2014,school,1,[learning]
7528,AZE,2014,school,1,[education]
7530,ISL,2014,school,4,"[university, test, training, education]"


In [8]:
# now we will count how often each country names other countries using country_list as check
records = []
for country_speech_year, speech_tokens in text.items():
    if "DS_Store" in country_speech_year:
        continue  
    
    country = country_speech_year[:3]
    year = int(country_speech_year[-8:-4])
    
    speech_str = " ".join(speech_tokens)
    
    for other_country in list_of_countries:
        if other_country.lower() in speech_str.lower():
            records.append({
                "country": country,
                "year": year,
                "mentioned_country": other_country
            })  

df_mentions = pd.DataFrame(records)

In [15]:
# rework the df such that it lists how often a country is mentioned by another one (instead of who's mentioning whom)
# df_mentions_count = df_mentions.groupby(['mentioned_country', 'country'])
df_mentions[df_mentions["year"]==2020]

,country,year,mentioned_country
32992,ISL,2020,Belarus
32993,ISL,2020,Georgia
32994,ISL,2020,Iceland
32995,ISL,2020,Libya
32996,ISL,2020,Ukraine
...,...,...,...
33916,AND,2020,World
33917,BHR,2020,Bahrain
33918,BHR,2020,Israel
33919,BHR,2020,Libya


In [ ]:
from numpy import disp


df_climate_topic = df_climate.groupby(['country', 'topic']).sum('mention_count').drop(columns=['year'])

df_climate_total = df_climate.groupby(['country']).sum('mention_count').drop(columns=['year'])


df_pivot = df_climate_topic.pivot_table(
    index='country',
    columns='topic',
    values='mention_count',
    aggfunc='sum',
    fill_value=0
)
df_pivot



topic,adaptation_and_nature_solutions,energy_and_technology,extreme_weather_and_impacts,greenhouse_gases_and_emissions,policy_agreements_and_economics
country,,,,,
AFG,0,2,1,0,0
AGO,0,2,0,0,1
ALB,0,4,0,0,0
AND,0,2,0,0,2
ARE,0,0,0,0,2
...,...,...,...,...,...
YMD,0,1,0,0,0
YUG,0,0,0,2,0
ZAF,0,0,0,0,2


In [ ]:
display(df_climate_total, df_climate_topic, df_climate_total[df_climate_total['mention_count'] >= 10])

,mention_count
country,
AFG,3
AGO,3
ALB,4
AND,4
ARE,2
...,...
YMD,1
YUG,2
ZAF,2


mention_count
country topic                                         
AFG     energy_and_technology                        2
        extreme_weather_and_impacts                  1
AGO     energy_and_technology                        2
        policy_agreements_and_economics              1
ALB     energy_and_technology                        4
...                                                ...
YUG     greenhouse_gases_and_emissions               2
ZAF     policy_agreements_and_economics              2
ZMB     energy_and_technology                        2
        policy_agreements_and_economics              3
ZWE     policy_agreements_and_economics              1

[320 rows x 1 columns]

,mention_count
country,
BFA,10
DMA,12
FJI,12
FSM,26
ISL,25
KEN,13
KHM,11
LCA,10
MDG,10
